# DocLayout-YOLO Bulk Layout Parsing (Dataset Builder)

This notebook is a standalone utility to:

- run DocLayout-YOLO layout parsing over a folder of PDFs
- export per-PDF layout JSON
- optionally export cropped visuals (figure/table) for caption-training datasets
- optionally upload artifacts to Cloudflare R2

It reuses the same DocLayout-YOLO approach as your Synapse `layout_worker.py`, but **does not** require Supabase or the queue workers.


In [ ]:
# Install dependencies
!pip -q install pymupdf pillow numpy doclayout-yolo huggingface-hub opencv-python-headless boto3


In [ ]:
# Optional: mount Google Drive (if your PDFs are stored there)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ---- Config ----
# Point INPUT_DIR to a folder containing PDFs.
INPUT_DIR = "/content/drive/MyDrive/visually_rich_pdfs"  # change me
OUTPUT_DIR = "/content/output_layout"  # local output folder

# Rendering / inference knobs
LAYOUT_RENDER_SCALE = 1.25
LAYOUT_IMGSZ = 768
LAYOUT_CONF = 0.2
LAYOUT_BATCH_PAGES = 16
LAYOUT_HALF = True
LAYOUT_RENDER_CHUNK_PAGES = 8
LAYOUT_PIPELINE_QUEUE_CHUNKS = 6

# Export crops for figure/table blocks
EXPORT_CROPS = True
CROPS_DIR = f"{OUTPUT_DIR}/crops"

# Optional: upload to R2 (set env vars below)
R2_UPLOAD = False
R2_PREFIX = "dataset_layout"  # folder prefix in the bucket

# DocLayout-YOLO weights
# These match the defaults used in Synapse. Override if you want a different checkpoint.
DOCLAYOUT_YOLO_REPO = "juliozhao/DocLayout-YOLO-DocStructBench"
DOCLAYOUT_YOLO_FILENAME = "doclayout_yolo_docstructbench_imgsz1024.pt"


## R2 (optional)

If you want to upload outputs to R2, set these env vars in Colab (Runtime -> Secrets or manually):

- `R2_ENDPOINT`
- `R2_BUCKET`
- `R2_ACCESS_KEY`
- `R2_SECRET_KEY`


In [ ]:
%%writefile doclayout_runner.py
import io
import json
import os
import queue
import tempfile
import threading
from dataclasses import dataclass
from pathlib import Path

import boto3
import fitz  # PyMuPDF
import numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download
from doclayout_yolo import YOLOv10


@dataclass
class RunnerConfig:
    input_dir: str
    output_dir: str
    crops_dir: str
    export_crops: bool
    render_scale: float
    render_chunk_pages: int
    pipeline_queue_chunks: int
    imgsz: int
    conf: float
    batch_pages: int
    half: bool
    device: str
    r2_upload: bool
    r2_prefix: str
    repo_id: str
    weights_filename: str


def _maybe_r2_client(cfg: RunnerConfig):
    if not cfg.r2_upload:
        return None
    endpoint = os.getenv("R2_ENDPOINT")
    bucket = os.getenv("R2_BUCKET")
    access = os.getenv("R2_ACCESS_KEY")
    secret = os.getenv("R2_SECRET_KEY")
    if not endpoint or not bucket or not access or not secret:
        raise RuntimeError("R2_UPLOAD=True but missing R2 env vars")
    s3 = boto3.client(
        "s3",
        endpoint_url=endpoint,
        aws_access_key_id=access,
        aws_secret_access_key=secret,
    )
    return s3


def _put_r2_json(s3, bucket: str, key: str, payload: dict):
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=json.dumps(payload).encode("utf-8"),
        ContentType="application/json",
    )


def _put_r2_png(s3, bucket: str, key: str, image: Image.Image):
    buf = io.BytesIO()
    image.save(buf, format="PNG", optimize=True)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue(), ContentType="image/png")


def load_model(cfg: RunnerConfig):
    weights_path = hf_hub_download(repo_id=cfg.repo_id, filename=cfg.weights_filename)
    model = YOLOv10(weights_path)
    return model


def _render_page_image(page: fitz.Page, render_scale: float) -> Image.Image:
    if render_scale <= 0:
        render_scale = 1.0
    pix = page.get_pixmap(matrix=fitz.Matrix(render_scale, render_scale))
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")


def _crop_bbox(img: Image.Image, bbox_img):
    try:
        x1, y1, x2, y2 = [float(v) for v in bbox_img]
    except Exception:
        return None
    w, h = img.size
    x1 = max(0, min(w, int(x1)))
    x2 = max(0, min(w, int(x2)))
    y1 = max(0, min(h, int(y1)))
    y2 = max(0, min(h, int(y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    return img.crop((x1, y1, x2, y2))


def _predict(model, inputs, cfg: RunnerConfig):
    kwargs = {"imgsz": cfg.imgsz, "conf": cfg.conf, "device": cfg.device}
    if cfg.half:
        kwargs["half"] = True
    try:
        return model.predict(inputs, **kwargs)
    except TypeError:
        kwargs.pop("half", None)
        return model.predict(inputs, **kwargs)


def detect_layout_pil(model, page_indices, pil_images, cfg: RunnerConfig):
    results = []
    for offset in range(0, len(pil_images), cfg.batch_pages):
        chunk_imgs = pil_images[offset : offset + cfg.batch_pages]
        chunk_page_indices = page_indices[offset : offset + cfg.batch_pages]
        det_res = _predict(model, chunk_imgs, cfg)
        for page_index, r in zip(chunk_page_indices, det_res):
            blocks = []
            boxes = getattr(r, "boxes", None)
            names = getattr(r, "names", None)
            if boxes is not None and getattr(boxes, "xyxy", None) is not None:
                xyxy = boxes.xyxy
                confs = getattr(boxes, "conf", None)
                clss = getattr(boxes, "cls", None)
                try:
                    xyxy = xyxy.cpu().numpy()
                    confs = confs.cpu().numpy() if confs is not None else None
                    clss = clss.cpu().numpy().astype(int) if clss is not None else None
                except Exception:
                    pass
                for i, bb in enumerate(xyxy):
                    cls_id = int(clss[i]) if clss is not None else -1
                    label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
                    score = float(confs[i]) if confs is not None else 1.0
                    x1, y1, x2, y2 = [float(v) for v in bb]
                    blocks.append({"type": label, "score": score, "bbox": [x1, y1, x2, y2]})
            results.append({"page": int(page_index), "blocks": blocks})
    return results


def detect_layout_for_pdf_streaming(model, pdf_bytes: bytes, cfg: RunnerConfig):
    q: queue.Queue = queue.Queue(maxsize=max(1, cfg.pipeline_queue_chunks))
    sentinel = object()

    def producer():
        try:
            doc = fitz.open(stream=pdf_bytes, filetype="pdf")
            n = doc.page_count
            for offset in range(0, n, cfg.render_chunk_pages):
                end = min(n, offset + cfg.render_chunk_pages)
                idxs = []
                imgs = []
                for i in range(offset, end):
                    page = doc.load_page(i)
                    img = _render_page_image(page, render_scale=cfg.render_scale)
                    idxs.append(i)
                    imgs.append(img)
                q.put((idxs, imgs))
        finally:
            q.put(sentinel)

    t = threading.Thread(target=producer, daemon=True)
    t.start()

    layout_all = []
    while True:
        item = q.get()
        if item is sentinel:
            break
        idxs, imgs = item
        layout_all.extend(detect_layout_pil(model, idxs, imgs, cfg))
    t.join(timeout=30)
    return layout_all


def run(cfg: RunnerConfig):
    in_dir = Path(cfg.input_dir)
    out_dir = Path(cfg.output_dir)
    crops_dir = Path(cfg.crops_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    if cfg.export_crops:
        crops_dir.mkdir(parents=True, exist_ok=True)

    # Model + device
    model = load_model(cfg)

    s3 = _maybe_r2_client(cfg)
    bucket = os.getenv("R2_BUCKET") if cfg.r2_upload else None

    pdfs = sorted(in_dir.rglob("*.pdf"))
    manifest_path = out_dir / "manifest.jsonl"
    with manifest_path.open("w", encoding="utf-8") as mf:
        for pdf_path in pdfs:
            rel = pdf_path.relative_to(in_dir)
            safe_stem = str(rel).replace("/", "_").replace("\\\\", "_")
            pdf_bytes = pdf_path.read_bytes()
            layout = detect_layout_for_pdf_streaming(model, pdf_bytes, cfg)

            out_json = {
                "source_pdf": str(rel),
                "render_scale": cfg.render_scale,
                "imgsz": cfg.imgsz,
                "conf": cfg.conf,
                "layout": layout,
            }
            out_key = f"{safe_stem}.layout.json"
            (out_dir / out_key).write_text(json.dumps(out_json), encoding="utf-8")

            # optional crops
            crops_written = 0
            if cfg.export_crops:
                doc = fitz.open(stream=pdf_bytes, filetype="pdf")
                img_cache = {}
                for page_info in layout:
                    pi = int(page_info.get("page") or 0)
                    blocks = page_info.get("blocks") or []
                    for bi, b in enumerate(blocks):
                        t = str(b.get("type") or "").lower()
                        if not ("figure" in t or "table" in t or "graph" in t or "chart" in t or "image" in t):
                            continue
                        bbox = b.get("bbox")
                        if not bbox:
                            continue
                        if pi not in img_cache:
                            img_cache[pi] = _render_page_image(doc.load_page(pi), render_scale=cfg.render_scale)
                        crop = _crop_bbox(img_cache[pi], bbox)
                        if crop is None:
                            continue
                        crop_name = f"{safe_stem}__p{pi}_b{bi}.png"
                        crop_path = crops_dir / crop_name
                        crop.save(crop_path)
                        crops_written += 1

            # optional upload
            r2_layout_key = None
            if cfg.r2_upload and s3 and bucket:
                r2_layout_key = f"{cfg.r2_prefix}/{out_key}"
                _put_r2_json(s3, bucket, r2_layout_key, out_json)

            mf.write(
                json.dumps(
                    {
                        "source_pdf": str(rel),
                        "layout_json": out_key,
                        "r2_layout_key": r2_layout_key,
                        "crops_written": crops_written,
                    }
                )
                + "\n"
            )

    return str(manifest_path), len(pdfs)


In [ ]:
import os
import torch
from doclayout_runner import RunnerConfig, run

device = "cuda:0" if torch.cuda.is_available() else "cpu"

cfg = RunnerConfig(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    crops_dir=CROPS_DIR,
    export_crops=EXPORT_CROPS,
    render_scale=float(LAYOUT_RENDER_SCALE),
    render_chunk_pages=int(LAYOUT_RENDER_CHUNK_PAGES),
    pipeline_queue_chunks=int(LAYOUT_PIPELINE_QUEUE_CHUNKS),
    imgsz=int(LAYOUT_IMGSZ),
    conf=float(LAYOUT_CONF),
    batch_pages=int(LAYOUT_BATCH_PAGES),
    half=bool(LAYOUT_HALF),
    device=device,
    r2_upload=bool(R2_UPLOAD),
    r2_prefix=R2_PREFIX,
    repo_id=DOCLAYOUT_YOLO_REPO,
    weights_filename=DOCLAYOUT_YOLO_FILENAME,
)

manifest, n = run(cfg)
print("Processed PDFs:", n)
print("Manifest:", manifest)
